# Lecture 07 companion notebook — R: the tidyverse and ggplot2

**BIOE 610 Computational Tools · Week 7**

Every example from Lecture 07 as cells you can run, change and break — packages, control flow, the pipe, `pivot_longer()`, ggplot2 built layer by layer, and the figure gallery (heatmap, growth curves, dose-response, PCA). There is a matching Python notebook (`Lecture07_Python_wrangling_figures.ipynb`) with the same sections in the same order — keep them open side by side to compare. Needs the **tidyverse** (`install.packages("tidyverse")` once); everything else is base R.

**How to use this notebook in Positron**

1. Open the file — Positron shows it as a notebook automatically
2. Pick the **R kernel** (top right). If no R kernel is offered, run this once in an R console, then reopen:
   ```r
   install.packages("IRkernel")
   IRkernel::installspec()
   ```
3. Run a cell with **Shift-Enter**. Run them in order the first time; then go back and experiment — *change the numbers, break things on purpose, read the errors*.

## 1 · Installing vs. loading packages

`install.packages()` puts a package on your computer — **once**. `library()` makes it available in *this* session — **every** session. "there is no package called ..." means you never installed it.

In [ ]:
library(tidyverse)   # loads ggplot2, dplyr, tidyr, readr, stringr, purrr, ...
# use one function without loading its package: readr::read_tsv, readxl::read_excel

## 2 · Inspecting objects

When an object surprises you, interrogate it first. These five answer most questions.

In [ ]:
stickle <- read_tsv("https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
                    show_col_types = FALSE) |>
  rename(geno_micro = `Geno&Micro`)       # fix the awkward name once

class(stickle)
dim(stickle)
names(stickle)[1:6]

In [ ]:
str(stickle[, 1:5])        # structure: type, dimensions, first values
summary(stickle$Gene1)     # quick summary of one column

## 3 · Logical operators and membership

Comparisons return logical vectors; logical vectors select rows. New today: **membership** with `%in%`.

In [ ]:
x <- c(1, 5, 10, 15, 20)
x > 10                  # one TRUE/FALSE per element
x[x > 10]               # keep only where TRUE - L05's logical indexing
x %in% c(1, 15, 99)     # membership: is each element in the set?

## 4 · Control flow: making decisions with `if`

R wraps blocks in `{ }`. The vectorized cousin `ifelse()` gives one answer per element — no loop needed.

In [ ]:
mass <- 1.42
if (mass > 1.0) {
  cat("big fish\n")
} else if (mass > 0.5) {
  cat("medium fish\n")
} else {
  cat("small fish\n")
}

In [ ]:
x <- c(0.3, 0.8, 1.42)
ifelse(x > 1.0, "big", "small")      # vectorized: one answer per element

## 5 · Control flow: repeating with `for` (and `while`)

Both languages loop directly over a collection's *elements* — no counters required. `break` leaves early; `next` skips to the next turn.

In [ ]:
genes <- c("tp53", "brca1", "myc")
for (g in genes) {
  cat(toupper(g), "\n")
}

for (i in seq_along(genes)) {        # ... or over the positions 1, 2, 3
  cat(i, genes[i], "\n")
}

In [ ]:
i <- 1
while (i < 8) {                      # repeat until the condition fails
  i <- i * 2
}
i

## 6 · When you do *not* need a loop

Vectorized code is shorter, faster and easier to read. Loops earn their keep for files, simulations, and step-*n*-depends-on-step-*n−1* problems.

In [ ]:
x <- c(2, 3, 4, 5)
x^2                                          # not: for (i in seq_along(x)) ...
genes_cols <- grep("^Gene", names(stickle))
head(rowSums(stickle[, genes_cols]))         # not: nested loops

## 7 · The pipe `|>`: "and then"

Each verb takes a data frame and returns a data frame, so they chain. Read `|>` as **"and then"**.

In [ ]:
mydata <- tibble(
  hydrogel_concentration = factor(c("low", "high", "high", "high",
                                    "medium", "medium", "medium", "low")),
  compression  = c(3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5),
  conductivity = c(0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3))

mydata |>
  filter(compression > 4) |>                         # keep rows
  mutate(ratio = conductivity / compression) |>      # add a column
  group_by(hydrogel_concentration) |>                # define groups
  summarize(n = n(), mean_ratio = mean(ratio))       # one row per group

## 8 · Wide to long: `pivot_longer()`

One `pivot_longer()` unlocks *every* gene at once: 80 × 600 = 48,000 rows, one per fish × gene.

In [ ]:
stickle_long <- stickle |>
  pivot_longer(starts_with("Gene"), names_to = "gene", values_to = "count")
stickle_long

In [ ]:
gene_means <- stickle_long |>
  group_by(gene, Genotype, Microbiota) |>
  summarize(mean_count = mean(count), .groups = "drop")
gene_means |> filter(gene == "Gene1")

## 9 · ggplot2, layer 1: data + aesthetics + a geom

`ggplot(data, aes(...))` declares the **mapping**; a **geom** draws it. `geom_jitter()` is `geom_point()` plus sideways noise.

In [ ]:
gene1 <- stickle_long |> filter(gene == "Gene1")

ggplot(gene1, aes(x = Genotype, y = count)) +
  geom_jitter(width = 0.08)

## 10 · Layer 2: more aesthetics, more geoms

Mapping a column to `fill` / `color` splits the groups **and** builds the legend — automatically, from the data. Layers draw in order.

In [ ]:
ggplot(gene1, aes(x = Genotype, y = count, fill = Microbiota)) +
  geom_violin(alpha = 0.4, color = NA, position = position_dodge(0.8)) +
  geom_point(aes(color = Microbiota), alpha = 0.6,
             position = position_jitterdodge(jitter.width = 0.08, dodge.width = 0.8))

## 11 · Layers 3–5: facets, scales, labels, theme — and `ggsave()`

`facet_wrap(~ gene)` gives one panel per gene for free. This is the left half of L06's destination figure.

In [ ]:
stickle_long |>
  filter(gene %in% paste0("Gene", 1:4)) |>
  ggplot(aes(Genotype, count, fill = Microbiota)) +
  geom_violin(alpha = 0.4, color = NA, position = position_dodge(0.8)) +
  geom_point(aes(color = Microbiota), alpha = 0.6, size = 1.2,
             position = position_jitterdodge(jitter.width = 0.08, dodge.width = 0.8)) +
  scale_fill_viridis_d(end = 0.7) + scale_color_viridis_d(end = 0.7) +
  facet_wrap(~ gene, scales = "free_y") +
  labs(x = "Host genotype", y = "Read count",
       title = "Stickleback gut RNA-seq",
       subtitle = "Each point is one fish; violins show the full distribution") +
  theme_minimal(base_size = 14) + theme(legend.position = "top")

In [ ]:
dir.create("results", showWarnings = FALSE)
ggsave("results/gene_panel.svg", width = 8, height = 6)   # saves the last plot drawn
list.files("results")

## 12 · The closer: all 600 genes in one figure

`group_by(gene) |> summarize()` collapses 48,000 rows to 600 — one point per gene. Dashed line: variance = mean (Poisson); real RNA-seq runs hotter.

In [ ]:
stickle_long |>
  group_by(gene) |>
  summarize(mean = mean(count), var = var(count)) |>
  ggplot(aes(mean, var)) +
  geom_point(alpha = 0.4, color = "#31688e") +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
  geom_smooth(se = FALSE, color = "#440154") +
  scale_x_log10() + scale_y_log10() +
  labs(x = "Mean count per gene (log)", y = "Variance (log)",
       title = "All 600 genes at once") +
  theme_minimal(base_size = 14)

# The figure gallery

## 13 · A clustered heatmap, from synthesized expression data

`scale = "row"` standardizes each gene, so the colors compare *patterns*, not levels. The dendrograms should rediscover the two modules and the control/treated split we build in.

In [ ]:
set.seed(2026)
expr <- matrix(rnorm(30 * 12, mean = 8, sd = 0.4), nrow = 30) +
        rnorm(30)                                 # a baseline level per gene
expr[1:15, 7:12]  <- expr[1:15, 7:12] + 2         # module 1: up in treated
expr[16:30, 7:12] <- expr[16:30, 7:12] - 1.5      # module 2: down in treated
rownames(expr) <- paste0("gene_", 1:30)
colnames(expr) <- c(paste0("ctrl_", 1:6), paste0("treat_", 1:6))

heatmap(expr, scale = "row", col = hcl.colors(50, "viridis"), margins = c(6, 5))

## 14 · Growth curves with ribbons

The `group_by() |> summarize()` + ribbon pattern works for any repeated time course: sensor logs, qPCR, cyclic loading.

In [ ]:
params <- tibble(strain = c("wild type", "mutant A", "mutant B"),
                 K = c(1.8, 1.2, 1.6),            # carrying capacity
                 r = c(0.25, 0.35, 0.15))         # growth rate
growth <- expand_grid(params, rep = 1:4, time = seq(0, 48, by = 2)) |>
  mutate(od = K / (1 + exp(-r * (time - 20))) + rnorm(n(), sd = 0.05))

growth |>
  group_by(strain, time) |>
  summarize(mean = mean(od), sd = sd(od), .groups = "drop") |>
  ggplot(aes(time, mean, color = strain, fill = strain)) +
  geom_ribbon(aes(ymin = mean - sd, ymax = mean + sd), alpha = 0.25, color = NA) +
  geom_line(linewidth = 1) +
  scale_color_viridis_d(end = 0.8) + scale_fill_viridis_d(end = 0.8) +
  labs(x = "Time (h)", y = "OD600",
       title = "Bioreactor growth curves (mean ± SD of 4 replicates)") +
  theme_minimal(base_size = 14)

## 15 · Dose-response curves

The log-scaled x axis is what makes dose-response curves readable — doses span five orders of magnitude. The IC50 is where each curve crosses 50%.

In [ ]:
dr <- expand_grid(tibble(compound = c("compound X", "compound Y"),
                         ic50 = c(3e-7, 5e-6)),
                  rep = 1:3,
                  dose = 10^seq(-9, -4, length.out = 8)) |>
  mutate(viability = 100 / (1 + (dose / ic50)^1.2) + rnorm(n(), sd = 4))

ggplot(dr, aes(dose, viability, color = compound)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_smooth(se = FALSE) +
  scale_x_log10() +
  scale_color_viridis_d(end = 0.6) +
  labs(x = "Dose (M)", y = "Viability (%)",
       title = "Dose-response: two compounds, three replicates") +
  theme_minimal(base_size = 14)

## 16 · PCA of the real stickleback fish

PCA compresses 600 correlated gene columns into a few axes of maximal variation. `log1p()` tames the counts; `scale. = TRUE` weights every gene equally.

In [ ]:
gene_mat <- stickle |> select(starts_with("Gene"))
pca <- prcomp(log1p(gene_mat), scale. = TRUE)
summary(pca)$importance[, 1:3]           # variance explained by PC1-PC3

In [ ]:
bind_cols(as_tibble(pca$x[, 1:2]),
          stickle |> select(geno_micro)) |>
  ggplot(aes(PC1, PC2, color = geno_micro)) +
  geom_point(size = 2.5, alpha = 0.8) +
  scale_color_viridis_d(name = "Group") +
  labs(title = "PCA: 80 fish from 600 genes",
       subtitle = "Distance between points = overall expression similarity") +
  theme_minimal(base_size = 14)

## Your turn

1. Rebuild the section-11 figure for four *different* genes and `ggsave()` it. A working script version is on the Examples page — but only look after your own runs.
2. Color the mean-variance plot: compute each gene's mean separately for genotypes A and B, and color each point by which genotype is higher.
3. Adapt the gallery heatmap to the real data: take the 30 most *variable* stickleback genes and cluster the 80 fish (`scale = "row"` matters!).